In [ ]:
from transformers import GPT2Config, GPT2LMHeadModel, AutoTokenizer, Trainer, TrainingArguments
from datasets import load_dataset
import torch
import wandb

In [ ]:
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()

my_secret = user_secrets.get_secret("wandb_api_key") 

wandb.login(key=my_secret)
wandb.init(project="tinystories-1m", name="gpt-neo-1m-training")

In [ ]:
config = GPT2Config(
    vocab_size=50257,
    n_positions=512,
    n_embd=64,
    n_layer=2,
    n_head=2,
    n_inner=64,
    resid_pdrop=0.1,
    embd_pdrop=0.1,
    attn_pdrop=0.1,
)

custom_gpt2_model = GPT2LMHeadModel(config)

print(f"Number of parameters: {sum(p.numel() for p in custom_gpt2_model.parameters())}")

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("gpt2")

tokenizer.add_special_tokens({'pad_token': '[PAD]'})
custom_gpt2_model.resize_token_embeddings(len(tokenizer))

In [ ]:
# Load the dataset
dataset = load_dataset("roneneldan/TinyStories")

# Tokenize the dataset
def tokenize_function(examples):
    # Tokenize the input text
    tokenized_output = tokenizer(
        examples['text'],
        truncation=True,
        padding="max_length",
        max_length=128,
    )
    # Add labels for causal language modeling
    tokenized_output['labels'] = tokenized_output['input_ids'].copy()
    return tokenized_output

tokenized_datasets = dataset.map(tokenize_function, batched=True, remove_columns=["text"])

# Split into training and evaluation sets
train_dataset = tokenized_datasets['train'].shuffle(seed=42)
eval_dataset = tokenized_datasets['validation'].shuffle(seed=42)

In [ ]:
# train_dataset = train_dataset.select(range(10000))
# eval_dataset = eval_dataset.select(range(1000))

In [ ]:
training_args = TrainingArguments(
    output_dir="./results",
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=4,
    logging_dir="./logs",
    logging_steps=100,
    save_steps=500,
    eval_strategy="steps",
    eval_steps=500,
    save_total_limit=2,
    report_to="wandb",
    fp16=True,  # Enable mixed precision training
)

In [ ]:
trainer = Trainer(
    model=custom_gpt2_model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    processing_class=tokenizer,
)

In [ ]:
trainer.train()

In [ ]:
custom_gpt2_model.save_pretrained("./custom-gpt2-tinystories")
tokenizer.save_pretrained("./custom-gpt2-tinystories")

In [ ]:
# Evaluate the model on the validation set
eval_results = trainer.evaluate()

# Print evaluation results
print(f"Evaluation Loss: {eval_results['eval_loss']}")
print(f"Perplexity: {torch.exp(torch.tensor(eval_results['eval_loss']))}")

In [ ]:
from transformers import TextStreamer

# Function to generate streaming outputs
def generate_streaming_output(model, tokenizer, prompt, max_length=100, temperature=0.7, top_k=50, top_p=0.95):
    # Set the model to evaluation mode
    model.eval()

    # Tokenize the input prompt
    input_ids = tokenizer.encode(prompt, return_tensors="pt").to(model.device)

    # Create a streamer object
    streamer = TextStreamer(tokenizer, skip_prompt=True)  # Skip the prompt to only stream new tokens

    # Generate text with streaming
    with torch.no_grad():  # Disable gradient calculation for faster inference
        output = model.generate(
            input_ids,
            max_length=max_length,
            temperature=temperature,
            top_k=top_k,
            top_p=top_p,
            do_sample=True,  # Enable sampling
            streamer=streamer,  # Pass the streamer for real-time output
        )

    # Decode the full output (optional)
    full_output = tokenizer.decode(output[0], skip_special_tokens=True)
    return full_output

# Example usage
prompt = "Once upon a time"
generated_text = generate_streaming_output(custom_gpt2_model, tokenizer, prompt)
print("\nFull Generated Text:")
print(generated_text)